In [7]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os
import urllib3

# SSL 경고 무시 (회사 네트워크/방화벽 문제 해결)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

API_KEY = "4ec6f1f8-5b10-4f89-a199-4c16ab1a8847"
BASE_URL = "https://content.guardianapis.com/search"

def fetch_day(date, api_key, max_retries=5):
    """
    특정 날짜(YYYY-MM-DD)의 모든 헤드라인을 Guardian API에서 가져온다.
    페이지네이션을 처리해서 모든 결과를 수집한다.
    429(Too Many Requests) 발생 시 점점 더 길게 기다리면서 재시도.
    """
    date_str = date.strftime("%Y-%m-%d")
    all_articles = []
    
    backoff = 5  # 첫 대기(초)
    
    for attempt in range(1, max_retries + 1):
        try:
            page = 1
            while True:
                params = {
                    "from-date": date_str,
                    "to-date": date_str,
                    "page": page,
                    "page-size": 200,
                    "api-key": api_key,
                    "show-fields": "headline"
                }
                
                # verify=False: SSL 인증서 검증 우회 (회사 네트워크 문제 해결)
                resp = requests.get(BASE_URL, params=params, timeout=30, verify=False)
                status = resp.status_code
                
                print(f"[INFO] {date_str} page {page} status: {status}")
                
                # 정상
                if status == 200:
                    data = resp.json()
                    response = data.get("response", {})
                    articles = response.get("results", [])
                    
                    if not articles:
                        break
                    
                    all_articles.extend(articles)
                    
                    # 페이지네이션 확인
                    total_pages = response.get("pages", 1)
                    if page >= total_pages:
                        break
                    
                    page += 1
                    time.sleep(1)  # API 요청 간 대기
                    
                # 레이트 리밋
                elif status == 429:
                    print(f"[WARN] {date_str} rate limited (429). {backoff}초 대기 후 재시도.")
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                
                # 그 외 에러
                else:
                    print(f"[WARN] {date_str} 요청 실패 (status {status})")
                    return []
            
            # 성공
            return all_articles
        
        except Exception as e:
            print(f"[ERROR] {date_str} 요청 중 예외: {e}")
            time.sleep(backoff)
            backoff *= 2
    
    # 모든 시도 실패
    print(f"[FAIL] {date_str} 최종 실패")
    return []


def extract_headlines_from_articles(articles, date):
    """
    articles에서 날짜와 헤드라인만 추출
    """
    rows = []
    date_str = date.strftime("%Y-%m-%d")
    
    for article in articles:
        # Guardian API에서 webTitle 또는 fields.headline 사용
        headline = article.get("webTitle", "") or article.get("fields", {}).get("headline", "")
        
        if headline:
            rows.append({
                "date": date_str,
                "headline": headline.strip()
            })
    
    return rows


def collect_headlines(start_date, end_date, output_filename):
    """
    지정된 기간의 헤드라인을 수집하고 CSV로 저장
    """
    all_rows = []
    
    current_date = start_date
    while current_date <= end_date:
        articles = fetch_day(current_date, API_KEY)
        
        if articles:
            day_rows = extract_headlines_from_articles(articles, current_date)
            print(f"[INFO] {current_date.strftime('%Y-%m-%d')}: {len(day_rows)}건 추출됨")
            all_rows.extend(day_rows)
        else:
            print(f"[INFO] {current_date.strftime('%Y-%m-%d')}: 데이터 없음")
        
        current_date += timedelta(days=1)
        
        # 일 사이 기본 대기 (API한테 숨 좀 쉬게)
        time.sleep(2)
    
    df = pd.DataFrame(all_rows)
    
    if not df.empty:
        df = (
            df
            .drop_duplicates(subset=["date", "headline"])
            .sort_values(["date", "headline"])
            .reset_index(drop=True)
        )
    
    print(f"\n[RESULT] 총 {len(df)}개 기사 헤드라인 수집 완료")
    print(f"\n첫 10개 헤드라인:")
    print(df.head(10))
    
    df.to_csv(output_filename, index=False, encoding="utf-8-sig")
    print(f"\n[DONE] CSV 저장 완료 → {output_filename}")
    print(f"파일 위치: {os.path.abspath(output_filename)}")
    
    return df

print("✅ 함수 정의 완료!")

✅ 함수 정의 완료!


In [ ]:
# 테스트: 2014년 1월 1주일 (2014-01-01 ~ 2014-01-07)
print("=== 테스트 실행: 2014-01-01 ~ 2014-01-07 (1주일) ===\n")

test_start = datetime(2014, 1, 1)
test_end = datetime(2014, 1, 7)

df_test = collect_headlines(test_start, test_end, "UK_news_test.csv")

print(f"\n✅ 테스트 완료! {len(df_test)}개 헤드라인 수집됨")
print("UK_news_test.csv 파일을 확인해보세요.")
print("\n정상적으로 동작하면 다음 셀을 실행하여 전체 데이터를 수집하세요.")

=== 테스트 실행: 2014-01-01 ~ 2014-01-07 (1주일) ===

[INFO] 2014-01-01 page 1 status: 200
[INFO] 2014-01-01: 176건 추출됨
[INFO] 2014-01-02 page 1 status: 429
[WARN] 2014-01-02 rate limited (429). 5초 대기 후 재시도.
[INFO] 2014-01-02 page 1 status: 429
[WARN] 2014-01-02 rate limited (429). 10초 대기 후 재시도.
[INFO] 2014-01-02 page 1 status: 429
[WARN] 2014-01-02 rate limited (429). 20초 대기 후 재시도.
[INFO] 2014-01-02 page 1 status: 429
[WARN] 2014-01-02 rate limited (429). 40초 대기 후 재시도.


In [ ]:
# 전체 데이터 수집: 2014-01-01 ~ 2025-12-31
# ⚠️ 주의: 약 4,000일 데이터 수집으로 시간이 오래 걸립니다 (약 2-3시간 이상)

print("=== 전체 데이터 수집: 2014-01-01 ~ 2025-12-31 ===\n")
print("⚠️ 이 작업은 시간이 오래 걸립니다. 중간에 멈추지 마세요!\n")

full_start = datetime(2014, 1, 1)
full_end = datetime(2025, 12, 31)

df_full = collect_headlines(full_start, full_end, "UK_news.csv")

print(f"\n🎉 전체 수집 완료! {len(df_full)}개 헤드라인 수집됨")
print("UK_news.csv 파일이 생성되었습니다.")

In [5]:
# 테스트 결과 확인 (커널에 저장된 변수 사용)
print(f"df_test 데이터 개수: {len(df_test)}")
print(f"\n첫 20개 헤드라인:")
print(df_test.head(20))
print(f"\n마지막 10개 헤드라인:")
print(df_test.tail(10))

df_test 데이터 개수: 0

첫 20개 헤드라인:
Empty DataFrame
Columns: []
Index: []

마지막 10개 헤드라인:
Empty DataFrame
Columns: []
Index: []


In [6]:
# API 응답 디버그 - 하루만 테스트
test_date = datetime(2014, 1, 1)
params = {
    "from-date": "2014-01-01",
    "to-date": "2014-01-01",
    "page": 1,
    "page-size": 10,
    "api-key": API_KEY,
    "show-fields": "headline"
}

resp = requests.get(BASE_URL, params=params)
print(f"Status: {resp.status_code}")
print(f"\nJSON Response:")
data = resp.json()
print(f"Response keys: {data.keys()}")
print(f"\nResponse status: {data.get('response', {}).get('status')}")
print(f"Total results: {data.get('response', {}).get('total')}")
print(f"Results count: {len(data.get('response', {}).get('results', []))}")

if data.get('response', {}).get('results'):
    print(f"\n첫 번째 기사 샘플:")
    first_article = data['response']['results'][0]
    print(f"Keys: {first_article.keys()}")
    print(f"webTitle: {first_article.get('webTitle')}")
    print(f"fields: {first_article.get('fields')}")

SSLError: HTTPSConnectionPool(host='content.guardianapis.com', port=443): Max retries exceeded with url: /search?from-date=2014-01-01&to-date=2014-01-01&page=1&page-size=10&api-key=4ec6f1f8-5b10-4f89-a199-4c16ab1a8847&show-fields=headline (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))